In [ ]:
 %pip install -q google-cloud-storage pandas ipywidgets


In [ ]:
#instructions: - download the BART_IMAGE LABELING label set (from the google sheet) as a .csv and upload it to /contents.
#              - upload credentials.json to /contents
#              - change the line "df_review = df[df['line_number'].between(681, 1020)].copy()" to your assigned part
#                   - BRANDON : 1-340
#                   - EMMA : 341-680
#                   - AKASH: 681-1020
#                   - KIARA : 1021-1359
#                   - GAVIN : 1360-1698
import pandas as pd
import os
from google.cloud import storage
from IPython.display import Image, display

# 1. SETUP CREDENTIALS
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/content/credentials.json'
client = storage.Client()

# 2. LOAD DATA
file_path = '/content/BART_IMAGE LABELING - BART_label_review_template.csv'

try:
    df = pd.read_csv(file_path, sep=None, engine='python')                      #this also accounts for the first row just being column headings and skips it.
    df['line_number'] = pd.to_numeric(df['line_number'], errors='coerce')
    df_review = df[df['line_number'].between(100, 240)].copy()
    df_review['line_number'] = df_review['line_number'].astype(int)

    print(f"Successfully loaded {len(df_review)} rows")
except Exception as e:
    print(f"Error: {e}")

# 3. IMAGE LOADER FUNCTION
def get_image_from_gcs(gs_path):
    try:
        parts = gs_path.replace("gs://", "").split("/")
        bucket_name = parts[0]
        blob_name = "/".join(parts[1:])
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        return blob.download_as_bytes()
    except:
        return None

# 4. REVIEW FUNCTION
def review_batch(dataframe, start_row=0, count=5):
    # Slice the dataframe based on the current viewing position
    subset = dataframe.iloc[start_row : start_row + count]

    for _, row in subset.iterrows():
        print("="*80)
        print(f"LINE: {row['line_number']} | DISC: {row['auto_discipline']} | PRIO: {row['auto_priority']}")
        print(f"DESC: {row['auto_description']}")

        img_bytes = get_image_from_gcs(row['image_uri'])
        if img_bytes:
            display(Image(data=img_bytes, width=600))
        else:
            print(f"Could not load image: {row['image_uri']}")

# 5. START REVIEWING
# Set the count to how many image/labels you want displayed
review_batch(df_review, start_row=0, count=100)